# Black Grouse 2030 Land-Use Scenario Project

## Stage 8: Comparing Alternative 2030 Restoration Scenarios

This notebook compares the 2024 baseline and the four equal-area 2030 restoration scenarios.

The comparison combines:

- core-habitat amount;
- habitat-patch configuration;
- edge density;
- habitat-network connectivity at multiple gap distances;
- restoration-location characteristics.

Because every 2030 scenario restores exactly 11.656 km², differences among scenarios represent differences in spatial configuration rather than habitat amount.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from scipy import ndimage

print("Packages imported successfully.")

Packages imported successfully.


In [2]:
PROJECT_DIR = Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

SCENARIO_RASTERS = {
    "baseline_2024": (
        PROCESSED_DIR / "LGN2024_harmonised_25m.tif"
    ),
    "dispersed": (
        PROCESSED_DIR / "landuse_2030_dispersed.tif"
    ),
    "patch_enlargement": (
        PROCESSED_DIR
        / "landuse_2030_patch_enlargement.tif"
    ),
    "connectivity_focused": (
        PROCESSED_DIR
        / "landuse_2030_connectivity_focused.tif"
    ),
    "integrated_low_matrix_pressure": (
        PROCESSED_DIR
        / "landuse_2030_integrated_low_matrix_pressure.tif"
    ),
}

CONNECTIVITY_TABLE_PATH = (
    TABLES_DIR
    / "core_habitat_connectivity_by_gap_distance.csv"
)

RESTORATION_SUMMARY_PATH = (
    TABLES_DIR
    / "2030_restoration_scenario_summary.csv"
)

CORE_HABITAT_CLASS = 2

for scenario_name, raster_path in SCENARIO_RASTERS.items():

    status = "FOUND" if raster_path.exists() else "MISSING"

    print(f"{scenario_name}: {status}")
    print(raster_path)

print(
    "\nConnectivity table:",
    "FOUND"
    if CONNECTIVITY_TABLE_PATH.exists()
    else "MISSING",
)

print(
    "Restoration summary:",
    "FOUND"
    if RESTORATION_SUMMARY_PATH.exists()
    else "MISSING",
)

baseline_2024: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\LGN2024_harmonised_25m.tif
dispersed: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_dispersed.tif
patch_enlargement: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_patch_enlargement.tif
connectivity_focused: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_connectivity_focused.tif
integrated_low_matrix_pressure: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_integrated_low_matrix_pressure.tif

Connectivity table: FOUND
Restoration summary: FOUND


In [3]:
scenario_arrays = {}
scenario_profiles = {}

reference_shape = None
reference_transform = None
reference_crs = None
reference_mask = None

for scenario_name, raster_path in SCENARIO_RASTERS.items():

    with rasterio.open(raster_path) as src:

        raster_data = src.read(1)

        scenario_arrays[scenario_name] = raster_data
        scenario_profiles[scenario_name] = src.profile.copy()

        current_mask = raster_data != 0

        if reference_shape is None:

            reference_shape = raster_data.shape
            reference_transform = src.transform
            reference_crs = src.crs
            reference_mask = current_mask.copy()

        else:

            if raster_data.shape != reference_shape:
                raise ValueError(
                    f"{scenario_name} has a different shape."
                )

            if src.transform != reference_transform:
                raise ValueError(
                    f"{scenario_name} has a different transform."
                )

            if src.crs != reference_crs:
                raise ValueError(
                    f"{scenario_name} has a different CRS."
                )

            if not np.array_equal(
                current_mask,
                reference_mask,
            ):
                raise ValueError(
                    f"{scenario_name} has a different analysis mask."
                )


connectivity_metrics = pd.read_csv(
    CONNECTIVITY_TABLE_PATH
)

restoration_summary = pd.read_csv(
    RESTORATION_SUMMARY_PATH
)


PIXEL_SIZE_METRES = abs(
    reference_transform.a
)

PIXEL_AREA_KM2 = (
    PIXEL_SIZE_METRES
    * PIXEL_SIZE_METRES
    / 1_000_000
)

ANALYSIS_AREA_KM2 = (
    reference_mask.sum()
    * PIXEL_AREA_KM2
)


print("All comparison rasters are aligned.")

print(
    "Scenarios in connectivity table:",
    sorted(
        connectivity_metrics["scenario"].unique()
    ),
)

print(
    "Scenarios in restoration summary:",
    sorted(
        restoration_summary["scenario"].unique()
    ),
)

print(f"Analysis area: {ANALYSIS_AREA_KM2:.2f} km²")

All comparison rasters are aligned.
Scenarios in connectivity table: ['baseline_2024', 'connectivity_focused', 'dispersed', 'integrated_low_matrix_pressure', 'patch_enlargement']
Scenarios in restoration summary: ['connectivity_focused', 'dispersed', 'integrated_low_matrix_pressure', 'patch_enlargement']
Analysis area: 762.88 km²


In [4]:
def calculate_landscape_metrics(
    raster_data,
    analysis_mask,
    scenario_name,
):
    """
    Calculate core-habitat amount and configuration metrics
    using eight-neighbour patch connectivity.
    """

    core_mask = (
        (raster_data == CORE_HABITAT_CLASS)
        & analysis_mask
    )

    connectivity_structure = np.ones(
        (3, 3),
        dtype=np.uint8,
    )

    labelled_patches, number_of_patches = ndimage.label(
        core_mask,
        structure=connectivity_structure,
    )

    patch_pixel_counts = np.bincount(
        labelled_patches.ravel()
    )[1:]

    patch_pixel_counts = patch_pixel_counts[
        patch_pixel_counts > 0
    ]

    patch_areas_km2 = (
        patch_pixel_counts
        * PIXEL_AREA_KM2
    )

    core_habitat_area_km2 = (
        core_mask.sum()
        * PIXEL_AREA_KM2
    )

    mean_patch_area_km2 = (
        patch_areas_km2.mean()
    )

    median_patch_area_km2 = np.median(
        patch_areas_km2
    )

    largest_patch_area_km2 = (
        patch_areas_km2.max()
    )

    patch_density_per_100_km2 = (
        number_of_patches
        / ANALYSIS_AREA_KM2
        * 100
    )

    largest_patch_index_percent = (
        largest_patch_area_km2
        / ANALYSIS_AREA_KM2
        * 100
    )

    horizontal_edges = np.sum(
        analysis_mask[:, :-1]
        & analysis_mask[:, 1:]
        & (
            core_mask[:, :-1]
            != core_mask[:, 1:]
        )
    )

    vertical_edges = np.sum(
        analysis_mask[:-1, :]
        & analysis_mask[1:, :]
        & (
            core_mask[:-1, :]
            != core_mask[1:, :]
        )
    )

    total_edge_length_metres = (
        horizontal_edges + vertical_edges
    ) * PIXEL_SIZE_METRES

    total_edge_length_km = (
        total_edge_length_metres / 1000
    )

    edge_density_m_per_ha = (
        total_edge_length_metres
        / (ANALYSIS_AREA_KM2 * 100)
    )

    return {
        "scenario": scenario_name,

        "core_habitat_area_km2": round(
            core_habitat_area_km2,
            3,
        ),

        "number_of_patches": int(
            number_of_patches
        ),

        "patch_density_per_100_km2": round(
            patch_density_per_100_km2,
            3,
        ),

        "mean_patch_area_km2": round(
            mean_patch_area_km2,
            4,
        ),

        "median_patch_area_km2": round(
            median_patch_area_km2,
            4,
        ),

        "largest_patch_area_km2": round(
            largest_patch_area_km2,
            3,
        ),

        "largest_patch_index_percent": round(
            largest_patch_index_percent,
            3,
        ),

        "total_edge_length_km": round(
            total_edge_length_km,
            3,
        ),

        "edge_density_m_per_ha": round(
            edge_density_m_per_ha,
            3,
        ),
    }

In [5]:
scenario_metric_records = []

for scenario_name, raster_data in scenario_arrays.items():

    scenario_metrics = calculate_landscape_metrics(
        raster_data=raster_data,
        analysis_mask=reference_mask,
        scenario_name=scenario_name,
    )

    scenario_metric_records.append(
        scenario_metrics
    )


scenario_landscape_metrics = pd.DataFrame(
    scenario_metric_records
)


SCENARIO_METRICS_OUTPUT = (
    TABLES_DIR
    / "2030_scenario_landscape_metrics.csv"
)

scenario_landscape_metrics.to_csv(
    SCENARIO_METRICS_OUTPUT,
    index=False,
)

display(scenario_landscape_metrics)

print("\nScenario landscape metrics saved:")
print(SCENARIO_METRICS_OUTPUT)

,scenario,core_habitat_area_km2,number_of_patches,patch_density_per_100_km2,mean_patch_area_km2,median_patch_area_km2,largest_patch_area_km2,largest_patch_index_percent,total_edge_length_km,edge_density_m_per_ha
0,baseline_2024,54.408,2570,336.883,0.0212,0.0012,13.093,1.716,1497.675,19.632
1,dispersed,66.063,19058,2498.181,0.0035,0.0006,13.101,1.717,3296.400,43.210
2,patch_enlargement,66.063,2290,300.180,0.0288,0.0031,13.418,1.759,1789.775,23.461
3,connectivity_focused,66.063,2546,333.738,0.0259,0.0012,14.749,1.933,1633.425,21.411
4,integrated_low_matrix_pressure,66.063,2574,337.408,0.0257,0.0012,13.878,1.819,1664.625,21.820



Scenario landscape metrics saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\2030_scenario_landscape_metrics.csv


,scenario,core_habitat_area_km2,number_of_patches,mean_patch_area_km2,largest_patch_area_km2,edge_density_m_per_ha,network_component_count_250m,largest_component_habitat_percent_250m,network_component_count_500m,largest_component_habitat_percent_500m,mean_suitability_score
0,baseline_2024,54.408,2570,0.0212,13.093,19.632,570.0,28.17,202.0,53.02,NaN
1,dispersed,66.063,19058,0.0035,13.101,43.210,67.0,99.42,4.0,99.99,0.4675
2,patch_enlargement,66.063,2290,0.0288,13.418,23.461,508.0,36.48,169.0,50.61,0.6748
3,connectivity_focused,66.063,2546,0.0259,14.749,21.411,557.0,47.76,199.0,53.54,0.7783
4,integrated_low_matrix_pressure,66.063,2574,0.0257,13.878,21.820,550.0,44.27,197.0,51.17,0.8382



Complete scenario-comparison table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\2030_scenario_comparison_complete.csv


In [7]:
COMPARISON_METRICS = [
    "core_habitat_area_km2",
    "number_of_patches",
    "patch_density_per_100_km2",
    "mean_patch_area_km2",
    "largest_patch_area_km2",
    "edge_density_m_per_ha",
    "network_component_count_250m",
    "largest_component_habitat_percent_250m",
    "network_component_count_500m",
    "largest_component_habitat_percent_500m",
]

baseline_row = (
    scenario_comparison[
        scenario_comparison["scenario"] == "baseline_2024"
    ]
    .iloc[0]
)

relative_change_records = []

for _, scenario_row in scenario_comparison.iterrows():

    scenario_name = str(
        scenario_row["scenario"]
    )

    if scenario_name == "baseline_2024":
        continue

    for metric_name in COMPARISON_METRICS:

        baseline_value = float(
            baseline_row[metric_name]
        )

        scenario_value = float(
            scenario_row[metric_name]
        )

        absolute_change = (
            scenario_value - baseline_value
        )

        if baseline_value != 0:
            percentage_change = (
                absolute_change
                / baseline_value
                * 100
            )
        else:
            percentage_change = np.nan

        relative_change_records.append(
            {
                "scenario": scenario_name,
                "metric": metric_name,
                "baseline_value": baseline_value,
                "scenario_value": scenario_value,
                "absolute_change": round(
                    absolute_change,
                    4,
                ),
                "percentage_change": round(
                    percentage_change,
                    2,
                ),
            }
        )


scenario_changes_from_baseline = pd.DataFrame(
    relative_change_records
)


SCENARIO_CHANGE_OUTPUT = (
    TABLES_DIR
    / "2030_scenario_changes_relative_to_2024.csv"
)

scenario_changes_from_baseline.to_csv(
    SCENARIO_CHANGE_OUTPUT,
    index=False,
)


percentage_change_table = (
    scenario_changes_from_baseline
    .pivot(
        index="scenario",
        columns="metric",
        values="percentage_change",
    )
    .reset_index()
)


display(
    percentage_change_table[
        [
            "scenario",
            "core_habitat_area_km2",
            "number_of_patches",
            "mean_patch_area_km2",
            "edge_density_m_per_ha",
            "network_component_count_250m",
            "largest_component_habitat_percent_250m",
        ]
    ]
)

print("\nBaseline-relative comparison saved:")
print(SCENARIO_CHANGE_OUTPUT)

metric,scenario,core_habitat_area_km2,number_of_patches,mean_patch_area_km2,edge_density_m_per_ha,network_component_count_250m,largest_component_habitat_percent_250m
0,connectivity_focused,21.42,-0.93,22.17,9.06,-2.28,69.54
1,dispersed,21.42,641.56,-83.49,120.10,-88.25,252.93
2,integrated_low_matrix_pressure,21.42,0.16,21.23,11.15,-3.51,57.15
3,patch_enlargement,21.42,-10.89,35.85,19.50,-10.88,29.50



Baseline-relative comparison saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\2030_scenario_changes_relative_to_2024.csv
